# NOAA Severe Weather Events

## Data needed:
- Zillow County level data with FIPS
- NOAA event details with FIPS

## Data cleaning
- Damage to crops and damage to property need to be cleaned of NA and put into numeric values
- deaths and injuries need to be cleaned of NA


## Steps
- Load Zillow data 
- Load NOAA files and merge together fatalities, details
- Flooding - boolean based on flood_cause
- Tornado - boolean based on tor_f_scale


## Analytical metrics
For each event id
- performance of the regional zillow index at months T+1, T+2, T+3, T+4, T+5, T+6
- performance of the national zillow index at months T+1, T+2, T+3, T+4, T+5, T+6
- damage to crops 
- damage to property
- flooding boolean
- tornado boolean
- duration of event END - BEGIN
- breadth of event RANGE 

In [1]:
import pandas as pd

In [2]:
noaa_2025_fatalities = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_fatalities-ftp_v1.0_d2025_c20260116.csv.gz"
noaa_2025_details = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2025_c20260116.csv.gz"
noaa_2025_locations = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_locations-ftp_v1.0_d2025_c20260116.csv.gz"

In [3]:
details = pd.read_csv(noaa_2025_details)
fatalities = pd.read_csv(noaa_2025_fatalities)
locations = pd.read_csv(noaa_2025_locations)

In [4]:
df = pd.merge(details, locations, on="EVENT_ID", how="left")

In [5]:
df.describe()

,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID_x,EVENT_ID,STATE_FIPS,YEAR,...,END_LAT,END_LON,YEARMONTH,EPISODE_ID_y,LOCATION_INDEX,RANGE,LATITUDE,LONGITUDE,LAT2,LON2
count,84962.000000,84962.000000,84962.000000,84962.000000,84962.000000,84962.000000,84962.000000,8.496200e+04,84962.000000,84962.0,...,62860.000000,62860.000000,49916.000000,49916.000000,49916.000000,49916.000000,49916.000000,49916.000000,4.991600e+04,4.991600e+04
mean,202505.218710,15.039194,1330.213778,202505.218710,16.041819,1463.119571,202831.651527,1.263086e+06,34.549387,2025.0,...,37.188646,-89.794612,202506.041710,203531.157865,1.892279,2.557469,37.249620,-89.968521,3.140101e+06,7.531612e+06
std,2.346468,8.645734,631.290816,2.346468,8.664111,607.885508,2723.143557,2.051732e+04,19.150606,0.0,...,5.465177,12.450918,1.884532,2379.604785,1.328562,5.270432,5.609071,13.038671,1.352863e+06,3.350370e+06
min,202501.000000,1.000000,0.000000,202501.000000,1.000000,0.000000,197521.000000,1.221835e+06,1.000000,2025.0,...,-14.338000,-170.832600,202501.000000,197604.000000,1.000000,0.000000,-14.338000,-170.833900,-1.420280e+06,-1.711839e+07
25%,202503.000000,7.000000,936.000000,202503.000000,8.000000,1120.000000,200493.000000,1.245701e+06,20.000000,2025.0,...,34.185600,-97.089100,202505.000000,201886.000000,1.000000,0.750000,33.997975,-97.490000,3.120139e+06,7.630600e+06
50%,202505.000000,15.000000,1500.000000,202505.000000,16.000000,1612.000000,202683.500000,1.264115e+06,36.000000,2025.0,...,37.560500,-88.039350,202506.000000,203296.000000,1.000000,1.430000,37.690000,-87.881000,3.626811e+06,8.358641e+06
75%,202507.000000,21.000000,1802.000000,202507.000000,23.000000,1900.000000,205112.000000,1.280578e+06,48.000000,2025.0,...,40.301100,-80.653000,202507.000000,205312.000000,3.000000,2.730000,40.348150,-80.195400,3.935978e+06,9.632508e+06
max,202510.000000,31.000000,2359.000000,202510.000000,31.000000,2359.000000,208578.000000,1.299986e+06,99.000000,2025.0,...,69.386900,171.062300,202510.000000,208578.000000,8.000000,199.310000,69.386900,171.306500,6.923214e+06,1.705003e+07


Parse the crop and property damage fields into numeric values

In [6]:
def parse_numeric_string(rawstring):
    if pd.isna(rawstring):
        return 0
    if "K" in rawstring:
        value = rawstring.replace("K","")
        value = float(value) * 1000
        return value
    if "M" in rawstring:
        value = rawstring.replace("M","")
        value = float(value) * 1000000
        return value
    if "B" in rawstring:
        value = rawstring.replace("B","")
        value = float(value) * 1000000000
        return value
    return float(rawstring)
df['DAMAGE_PROPERTY'] = df["DAMAGE_PROPERTY"].apply(parse_numeric_string)
df['DAMAGE_CROPS'] = df["DAMAGE_CROPS"].apply(parse_numeric_string)

In [7]:
def has_flooding(rawstring):
    if pd.isna(rawstring):
        return 0
    return 1

df["FLOODING"] = df["FLOOD_CAUSE"].apply(has_flooding)
df["FLOODING"]

0        0
1        0
2        0
3        0
4        0
        ..
84957    0
84958    1
84959    1
84960    1
84961    1
Name: FLOODING, Length: 84962, dtype: int64

In [8]:
def has_tornado(rawstring):
    if pd.isna(rawstring):
        return 0
    if len(rawstring) > 0:
        return 1

df["TORNADO"] = df["TOR_F_SCALE"].apply(has_tornado)
df["TORNADO"]

0        0
1        1
2        0
3        0
4        0
        ..
84957    0
84958    0
84959    0
84960    0
84961    0
Name: TORNADO, Length: 84962, dtype: int64

In [9]:

df["BEGIN_DATE_TIME"] = pd.to_datetime(df["BEGIN_DATE_TIME"])
df["END_DATE_TIME"] = pd.to_datetime(df["END_DATE_TIME"])
duration_delta = df["END_DATE_TIME"] - df["BEGIN_DATE_TIME"]
df["Duration_Hours"] = duration_delta.dt.total_seconds() / 3600.0
df["Duration_Hours"]

0         0.033333
1         0.050000
2        28.450000
3         6.000000
4         6.000000
           ...    
84957     0.000000
84958     1.533333
84959     1.533333
84960     1.533333
84961     1.533333
Name: Duration_Hours, Length: 84962, dtype: float64

In [10]:

df.loc[:, "RANGE"] 

0          NaN
1          NaN
2          NaN
3          NaN
4          NaN
         ...  
84957    12.15
84958     7.12
84959     7.05
84960     2.49
84961     3.32
Name: RANGE, Length: 84962, dtype: float64

In [ ]:
class NOAAStormParser:
    def __init__(self, details_url, locations_url):
        details_df = pd.read_csv(details_url)
        locations_df = pd.read_csv(locations_url)
        self.df = pd.merge(locations_df, details_df, on="EVENT_ID", how="left")
        self.clean()
        
    def clean(self):
        self.parse_damages()
        self.parse_storm_duration()
        
    def parse_storm_duration(self)
        self.df["BEGIN_DATE_TIME"] = pd.to_datetime(df["BEGIN_DATE_TIME"])
        self.df["END_DATE_TIME"] = pd.to_datetime(df["END_DATE_TIME"])
        duration_delta = self.df["END_DATE_TIME"] - self.df["BEGIN_DATE_TIME"]
        self.df["DURATION_HOURS"] = duration_delta.dt.total_seconds() / 3600.0
        
    def parse_damages(self):
        self.df['DAMAGE_PROPERTY'] = self.df["DAMAGE_PROPERTY"].apply(self.parse_numeric_string)
        self.df['DAMAGE_CROPS'] = self.df["DAMAGE_CROPS"].apply(self.parse_numeric_string)

    def parse_numeric_string(self, rawstring):
        if pd.isna(rawstring):
            return 0

        rawstring = str(rawstring) 
        
        if "K" in rawstring:
            value = rawstring.replace("K","")
            return float(value) * 1000
        if "M" in rawstring:
            value = rawstring.replace("M","")
            return float(value) * 1_000_000
        if "B" in rawstring:
            value = rawstring.replace("B","")
            return float(value) * 1_000_000_000
        return float(rawstring)
